# 04 — Classical ML Models

**Goal:** Build and run Logistic Regression and all four tree ensemble models (Random Forest, XGBoost, LightGBM, CatBoost) through a single expanding-window cross-validation loop.

**Modules used:** `src/models/logistic_regression.py`, `src/models/tree_models.py`, `src/models/model_factory.py`, `src/training/trainer.py` (sklearn path)

---

## 0 · Imports & pipeline setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from src.data.load_data import load_market_data
from src.data.preprocess import apply_missing_value_policy, select_columns
from src.data.labeling import compute_forward_return, compute_threshold, make_labels
from src.data.splitters import split_dev_test, make_expanding_folds
from src.data.sequence_builder import build_sequences, drop_neutral_sequences, flatten_sequences
from src.models.model_factory import build_model
from src.training.trainer import run_single_sklearn_fold
from src.evaluation.metrics import summarize_fold_metrics
from src.utils.seed import set_global_seed

set_global_seed(42)

DATA_PATH    = ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv'
FEATURE_COLS = ['VIX_Term_Structure','Yield_Curve','SKEW_Index',
                'Risk_Appetite_Ratio','Crude_Oil','VWAP_Deviation','Volume_Momentum']
DATE_COL     = 'Date'
CLOSE_COL    = 'Nasdaq_Close'
LOOKBACK     = 21
N_FOLDS      = 4
VAL_RATIO    = 0.10
TEST_RATIO   = 0.15

---
## 1 · Rebuild the data pipeline

In [ ]:
df_raw   = load_market_data(str(DATA_PATH), date_col=DATE_COL)
df_clean = apply_missing_value_policy(df_raw, method='ffill_then_drop_head')
df       = select_columns(df_clean, FEATURE_COLS, CLOSE_COL, DATE_COL)

dev_df, test_df = split_dev_test(df, test_ratio=TEST_RATIO)
folds = make_expanding_folds(len(dev_df), n_folds=N_FOLDS, val_ratio_within_dev=VAL_RATIO)

print(f'Dev: {len(dev_df)} rows | Test: {len(test_df)} rows | Folds: {len(folds)}')

---
## 2 · Model definitions

All models are built from the config dict via `build_model`. Here we define a minimal config that mirrors `configs/base.yaml`.

In [ ]:
BASE_CONFIG = {
    'logreg':    {'class_weight': 'balanced', 'max_iter': 2000, 'solver': 'lbfgs'},
    'rf':        {'n_estimators': 300, 'max_depth': 6, 'class_weight': 'balanced', 'n_jobs': -1},
    'xgboost':   {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05,
                  'subsample': 0.8, 'colsample_bytree': 0.8, 'use_label_encoder': False,
                  'eval_metric': 'logloss', 'verbosity': 0},
    'lightgbm':  {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05,
                  'subsample': 0.8, 'colsample_bytree': 0.8, 'verbose': -1},
    'catboost':  {'iterations': 300, 'depth': 4, 'learning_rate': 0.05,
                  'verbose': 0, 'auto_class_weights': 'Balanced'},
}

SKLEARN_MODELS = ['logreg', 'rf', 'xgboost', 'lightgbm', 'catboost']

for name in SKLEARN_MODELS:
    model = build_model(name, BASE_CONFIG)
    print(f'{name:<12} → {type(model).__name__}')

---
## 3 · Cross-validation loop

For every (model, fold) combination:
1. Slice dev_df into train / val rows.
2. Compute forward returns and threshold **on train rows only**.
3. Build sequences → drop neutral → flatten to 2D.
4. Fit `StandardScaler` on train features only → transform both.
5. Train and evaluate with `run_single_sklearn_fold`.

In [ ]:
all_results = {}  # model_name -> list of fold metric dicts

for model_name in SKLEARN_MODELS:
    fold_metrics = []
    print(f'\n=== {model_name.upper()} ===')

    for fold_idx, (tr_range, vl_range) in enumerate(folds):
        train_df = dev_df.iloc[tr_range].reset_index(drop=True)
        val_df   = dev_df.iloc[vl_range].reset_index(drop=True)

        # --- Labeling (threshold from train only) ---
        train_fwd = compute_forward_return(train_df[CLOSE_COL], horizon=1)
        val_fwd   = compute_forward_return(val_df[CLOSE_COL], horizon=1)
        threshold = compute_threshold(train_fwd, method='quantile', quantile=0.40)

        train_labels = make_labels(train_fwd, threshold)
        val_labels   = make_labels(val_fwd, threshold)

        # --- Sequences ---
        X_tr, y_tr, _ = build_sequences(train_df, FEATURE_COLS, train_labels, LOOKBACK)
        X_vl, y_vl, _ = build_sequences(val_df,   FEATURE_COLS, val_labels,   LOOKBACK)

        X_tr, y_tr, _ = drop_neutral_sequences(X_tr, y_tr, pd.Series(range(len(y_tr))))
        X_vl, y_vl, _ = drop_neutral_sequences(X_vl, y_vl, pd.Series(range(len(y_vl))))

        X_tr_flat = flatten_sequences(X_tr)
        X_vl_flat = flatten_sequences(X_vl)

        # --- Scaling (fit on train only) ---
        scaler = StandardScaler()
        X_tr_flat = scaler.fit_transform(X_tr_flat)
        X_vl_flat = scaler.transform(X_vl_flat)

        # --- Train & evaluate ---
        model = build_model(model_name, BASE_CONFIG)
        result = run_single_sklearn_fold(model, X_tr_flat, y_tr, X_vl_flat, y_vl)
        fold_metrics.append(result['val_metrics'])

        mcc = result['val_metrics']['mcc']
        f1  = result['val_metrics']['f1']
        print(f'  Fold {fold_idx+1} | val_mcc={mcc:.4f}  val_f1={f1:.4f}')

    all_results[model_name] = fold_metrics

print('\nCV complete.')

---
## 4 · Aggregate fold metrics

In [ ]:
rows = []
for model_name, folds_metrics in all_results.items():
    summary = summarize_fold_metrics(folds_metrics)
    rows.append({
        'model':              model_name,
        'mcc_mean':           round(summary['mcc_mean'], 4),
        'mcc_std':            round(summary['mcc_std'], 4),
        'f1_mean':            round(summary['f1_mean'], 4),
        'balanced_acc_mean':  round(summary['balanced_accuracy_mean'], 4),
        'roc_auc_mean':       round(summary.get('roc_auc_mean') or 0, 4),
    })

results_df = pd.DataFrame(rows).sort_values('mcc_mean', ascending=False).reset_index(drop=True)
results_df

---
## 5 · Visualise results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# MCC
ax = axes[0]
bars = ax.barh(results_df['model'], results_df['mcc_mean'],
               xerr=results_df['mcc_std'], capsize=4,
               color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xlabel('MCC (mean ± std over 4 folds)')
ax.set_title('Validation MCC', fontsize=12)
ax.axvline(0, color='black', linewidth=0.8)

# F1 vs Balanced Accuracy
ax2 = axes[1]
x = np.arange(len(results_df))
width = 0.35
ax2.bar(x - width/2, results_df['f1_mean'], width, label='F1', color='mediumseagreen')
ax2.bar(x + width/2, results_df['balanced_acc_mean'], width, label='Balanced Acc', color='mediumpurple')
ax2.set_xticks(x)
ax2.set_xticklabels(results_df['model'], rotation=15)
ax2.set_title('F1 vs Balanced Accuracy', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

---
## 6 · Per-fold MCC heatmap

In [ ]:
import seaborn as sns

heatmap_data = pd.DataFrame(
    {name: [m['mcc'] for m in folds_m] for name, folds_m in all_results.items()},
    index=[f'Fold {i+1}' for i in range(N_FOLDS)]
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, vmin=-0.3, vmax=0.5)
ax.set_title('MCC per Fold — Classical ML Models', fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

| Model | Role |
|---|---|
| Logistic Regression | Linear baseline — measures if a linear decision boundary is sufficient |
| Random Forest | Bagging ensemble — robust to noise via averaging |
| XGBoost | Boosting with regularisation — usually strong on tabular data |
| LightGBM | Fast gradient boosting — leaf-wise growth |
| CatBoost | Boosting with ordered statistics — handles imbalance well |

**Next:** `05_deep_learning_models.ipynb` — architecture walkthrough of the five sequence models.